In [ ]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

seed = 42

def set_seed(value):
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(value)
        torch.cuda.manual_seed_all(value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

data = pd.read_csv("supplementary_data/data.csv")
continuous_cols = ["O", "N", "SSA", "PV", "RMIC", "Dap", "ID/IG", "CD"]
categorical_cols = ["Anion"]
target_col = "Cs"
data["Anion"] = data["Anion"].map({"SO4": 0, "OTf": 1}).astype(int)

x_cont = data[continuous_cols].to_numpy(dtype=np.float32)
x_cat = data[categorical_cols].to_numpy(dtype=np.int64)
y = data[target_col].to_numpy(dtype=np.float32)

cs_bin = pd.qcut(y, q=10, labels=False, duplicates="drop")
x_cont_train_full, x_cont_test, x_cat_train_full, x_cat_test, y_train_full, y_test = train_test_split(
    x_cont,
    x_cat,
    y,
    test_size=0.2,
    random_state=seed,
    stratify=cs_bin,
)

train_bin = pd.qcut(y_train_full, q=10, labels=False, duplicates="drop")
x_cont_train, x_cont_val, x_cat_train, x_cat_val, y_train, y_val = train_test_split(
    x_cont_train_full,
    x_cat_train_full,
    y_train_full,
    test_size=0.15,
    random_state=seed,
    stratify=train_bin,
)

cont_scaler = StandardScaler()
x_cont_train_scaled = cont_scaler.fit_transform(x_cont_train).astype(np.float32)
x_cont_val_scaled = cont_scaler.transform(x_cont_val).astype(np.float32)
x_cont_test_scaled = cont_scaler.transform(x_cont_test).astype(np.float32)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel().astype(np.float32)
y_val_scaled = y_scaler.transform(y_val.reshape(-1, 1)).ravel().astype(np.float32)
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1)).ravel().astype(np.float32)

train_dataset = TensorDataset(
    torch.from_numpy(x_cont_train_scaled),
    torch.from_numpy(x_cat_train).long(),
    torch.from_numpy(y_train_scaled),
)
val_dataset = TensorDataset(
    torch.from_numpy(x_cont_val_scaled),
    torch.from_numpy(x_cat_val).long(),
    torch.from_numpy(y_val_scaled),
)
test_dataset = TensorDataset(
    torch.from_numpy(x_cont_test_scaled),
    torch.from_numpy(x_cat_test).long(),
    torch.from_numpy(y_test_scaled),
)

batch_size = 32
loader_generator = torch.Generator().manual_seed(seed)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=loader_generator,
)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

class TabTransformer(nn.Module):
    def __init__(
        self,
        categories,
        num_continuous,
        dim=32,
        depth=4,
        heads=4,
        attn_dropout=0.1,
        ff_dropout=0.1,
        mlp_hidden_dims=(128, 64),
    ):
        super().__init__()
        if dim % heads != 0:
            raise ValueError(f"dim={dim} must be divisible by heads={heads}.")
        self.embeds = nn.ModuleList(
            [nn.Embedding(num_categories, dim) for num_categories in categories]
        )
        self.cat_len = len(categories)
        self.cont_norm = nn.BatchNorm1d(num_continuous)
        self.cont_proj = nn.Linear(num_continuous, dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dropout=attn_dropout,
            dim_feedforward=dim * 4,
            activation="gelu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        mlp_layers = []
        input_dim = dim * (self.cat_len + 1)
        for hidden_dim in mlp_hidden_dims:
            mlp_layers.extend(
                [
                    nn.Linear(input_dim, hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(ff_dropout),
                ]
            )
            input_dim = hidden_dim
        mlp_layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*mlp_layers)

    def forward(self, x_cat, x_cont):
        cat_tokens = torch.stack(
            [embedding(x_cat[:, i]) for i, embedding in enumerate(self.embeds)],
            dim=1,
        )
        cont_token = self.cont_proj(self.cont_norm(x_cont)).unsqueeze(1)
        tokens = torch.cat([cat_tokens, cont_token], dim=1)
        encoded = self.transformer(tokens)
        flattened = encoded.reshape(encoded.size(0), -1)
        return self.mlp(flattened).squeeze(-1)

model = TabTransformer(
    categories=[int(data[column].nunique()) for column in categorical_cols],
    num_continuous=len(continuous_cols),
    dim=32,
    depth=4,
    heads=4,
    attn_dropout=0.1,
    ff_dropout=0.1,
    mlp_hidden_dims=(128, 64),
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=10,
)

best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(300):
    model.train()
    train_loss = 0.0
    for batch_x_cont, batch_x_cat, batch_y in train_loader:
        batch_x_cont = batch_x_cont.to(device)
        batch_x_cat = batch_x_cat.to(device)
        batch_y = batch_y.to(device)
        prediction = model(batch_x_cat, batch_x_cont)
        loss = criterion(prediction, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_y.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    val_predictions = []
    val_labels = []
    with torch.no_grad():
        for batch_x_cont, batch_x_cat, batch_y in val_loader:
            batch_x_cont = batch_x_cont.to(device)
            batch_x_cat = batch_x_cat.to(device)
            batch_y = batch_y.to(device)
            prediction = model(batch_x_cat, batch_x_cont)
            val_loss += criterion(prediction, batch_y).item() * batch_y.size(0)
            val_predictions.append(prediction.cpu().numpy())
            val_labels.append(batch_y.cpu().numpy())
    val_loss /= len(val_loader.dataset)
    val_predictions = np.concatenate(val_predictions)
    val_labels = np.concatenate(val_labels)
    val_r2 = r2_score(val_labels, val_predictions)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1:03d} | Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val R2: {val_r2:.4f}"
        )
    if epochs_without_improvement >= 30:
        print(f"Early stopping at epoch {epoch + 1}")
        break

model.load_state_dict(best_state)

def evaluate(loader):
    model.eval()
    predictions = []
    labels = []
    with torch.no_grad():
        for batch_x_cont, batch_x_cat, batch_y in loader:
            predictions.append(
                model(batch_x_cat.to(device), batch_x_cont.to(device)).cpu().numpy()
            )
            labels.append(batch_y.numpy())
    predictions = np.concatenate(predictions)
    labels = np.concatenate(labels)
    predictions_real = y_scaler.inverse_transform(predictions[:, None]).ravel()
    labels_real = y_scaler.inverse_transform(labels[:, None]).ravel()
    r2 = r2_score(labels_real, predictions_real)
    rmse = np.sqrt(mean_squared_error(labels_real, predictions_real))
    mae = mean_absolute_error(labels_real, predictions_real)
    mape = np.mean(np.abs((labels_real - predictions_real) / labels_real)) * 100
    return r2, rmse, mae, mape

train_r2, train_rmse, train_mae, train_mape = evaluate(train_loader)
test_r2, test_rmse, test_mae, test_mape = evaluate(test_loader)

print("=================== TabTransformer ===================")
print(f"Train | R2: {train_r2:.4f} | RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f} | MAPE: {train_mape:.2f}%")
print(f"Test  | R2: {test_r2:.4f} | RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f} | MAPE: {test_mape:.2f}%")
